# Parallelization with Multiprocessing
Multiprocessing : run tasks in parallel using multiple CPU cores. Transfer function maps density values to color (RGB) and opacity (alpha). It controls how the 3D volume will appear when rendered

In [15]:
import os
import multiprocessing
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import h5py as h5
from scipy.interpolate import interpn

In [16]:
def transferFunction(x):
    r = (
        1.0 * np.exp(-((x - 9.0) ** 2) / 1.0)
        + 0.1 * np.exp(-((x - 3.0) ** 2) / 0.1)
        + 0.1 * np.exp(-((x + 3.0) ** 2) / 0.5)
    )
    g = (
        1.0 * np.exp(-((x - 9.0) ** 2) / 1.0)
        + 1.0 * np.exp(-((x - 3.0) ** 2) / 0.1)
        + 0.1 * np.exp(-((x + 3.0) ** 2) / 0.5)
    )
    b = (
        0.1 * np.exp(-((x - 9.0) ** 2) / 1.0)
        + 0.1 * np.exp(-((x - 3.0) ** 2) / 0.1)
        + 1.0 * np.exp(-((x + 3.0) ** 2) / 0.5)
    )
    a = (
        1.0 * np.exp(-((x - 9.0) ** 2) / 1.0)
        + 0.2 * np.exp(-((x - 3.0) ** 2) / 0.1)
        + 0.05 * np.exp(-((x + 3.0) ** 2) / 0.5)
    )
    return r, g, b, a

In [17]:
def render_angle(i, datacube, points, Nangles):
    angle = np.pi / 2 * i / Nangles
    N = 180
    c = np.linspace(-N / 2, N / 2, N)
    qx, qy, qz = np.meshgrid(c, c, c)
    qxR = qx
    qyR = qy * np.cos(angle) - qz * np.sin(angle)
    qzR = qy * np.sin(angle) + qz * np.cos(angle)
    qi = np.array([qxR.ravel(), qyR.ravel(), qzR.ravel()]).T
    camera_grid = interpn(points, datacube, qi, method="linear").reshape((N, N, N))
    image = np.zeros((camera_grid.shape[1], camera_grid.shape[2], 3))
    for dataslice in camera_grid:
        r, g, b, a = transferFunction(np.log(dataslice))
        image[:, :, 0] = a * r + (1 - a) * image[:, :, 0]
        image[:, :, 1] = a * g + (1 - a) * image[:, :, 1]
        image[:, :, 2] = a * b + (1 - a) * image[:, :, 2]
    image = np.clip(image, 0.0, 1.0)
    plt.figure(figsize=(4, 4), dpi=80)
    plt.imshow(image)
    plt.axis("off")
    plt.savefig(f"volumerender{i}.png", dpi=240, bbox_inches="tight", pad_inches=0)
    plt.close()
    return f"volumerender{i}.png"

In [5]:
!python3 multiprocessing_rendering.py

Rendered images: ['volumerender0.png', 'volumerender1.png', 'volumerender2.png', 'volumerender3.png', 'volumerender4.png', 'volumerender5.png', 'volumerender6.png', 'volumerender7.png', 'volumerender8.png', 'volumerender9.png']
Figure(400x400)
Figure(400x400)
Figure(400x400)
Figure(400x400)
Figure(400x400)
Figure(400x400)
Figure(400x400)
Figure(400x400)
Figure(400x400)
Figure(400x400)


In this experiment, a 3D volumetric dataset stored in an HDF5 file is visualized using volume rendering techniques. The transfer function converts density values into color and opacity, allowing different structures inside the data to be visualized.

The program generates rendered images from multiple viewing angles by rotating the volume grid. Interpolation is used to obtain density values at rotated coordinates, and alpha compositing is applied to accumulate colors across slices.

Multiprocessing is used to render images in parallel across CPU cores. This significantly reduces total rendering time since each angle is processed independently. The final output is a series of images showing the volume from different perspectives, helping to better understand the internal structure of the dataset.